# 13 — Final Research Summary

**Purpose:** Aggregate all evidence into the formal go / revise / no-go recommendation (mandate §9/§10).

**Research questions:**
1. Which pairs and configurations survived every gate (02→12)?
2. What is the honest expected performance range, cited at fold-median under stressed costs?
3. What are the known failure modes and open risks?
4. Go, revise, or no-go — and why?

**Data used:** the banked machine-readable outputs of the notebook-02 and notebook-03 QuantConnect runs
(`reports/machine_readable/nb02_<PAIR>_*.csv`, seven pairs), the decision log, the assumptions register and
the limitations register. No market data is read here and no new statistic is computed: every number below
is recomputed from artifacts the pre-registered runs already emitted, so this notebook is a **cross-check**
of validation reports 02 and 03 rather than a re-typing of them.

**Answer, up front:** *no-go for Version 1 at intraday horizon.* Question 2 has no honest answer because
question 1's answer is "none".

In [1]:
import sys
sys.path.insert(0, "../src")
import numpy as np
import pandas as pd
import yaml

RESEARCH_CONFIG = yaml.safe_load(open("../config/research_config.yaml"))
SEED = RESEARCH_CONFIG["meta"]["random_seed"]
np.random.seed(SEED)
pd.set_option("display.width", 200, "display.max_columns", 60)
print(f"config loaded | global seed = {SEED}")

config loaded | global seed = 20260801


## Methodology

Evidence table: every pair in the locked universe with its verdict, its pre-registration, the evidence that
decided it, and the effect size in bps beside the t-statistic (D-010 requires both, because at n ≈ 680k bars
significance is free). Funnel accounting (pairs in → pairs surviving each gate); total-trials disclosure;
limitations register review.

Aggregation lives in `spread_research.program_summary`, which reads only banked CSVs. Its unit tests pin
every headline figure quoted in reports 02 and 03 against those same CSVs — if a report and its own evidence
ever drift apart, `tests/unit/test_program_summary.py` fails rather than this notebook quietly agreeing with
the prose.

**The recommendation may be negative. A LEAN build only proceeds if the user issues `PROCEED TO LEAN BUILD`
after reading it.**

In [2]:
from pathlib import Path
from spread_research.program_summary import (
    PAIRS, REGISTRY, largest_honest_effect, leg_vr, matched_elapsed_vr,
    program_table, program_totals, read_conditional, read_variance_ratio,
)

MR = Path("../reports/machine_readable")
EVIDENCE_AVAILABLE = (MR / "nb02_MES_MYM_scalars.csv").exists()
print("banked notebook-02/03 evidence:", "present" if EVIDENCE_AVAILABLE else "MISSING")

table = program_table(root=MR)
totals = program_totals(table)
totals

banked notebook-02/03 evidence: present


{'pairs_tested': 7,
 'pairs_reversion_present': 0,
 'pairs_effect_clears_cost': 1,
 'cells_examined': 420,
 'cells_honest': 185,
 'bars_min': 675840,
 'bars_max': 684300,
 'sessions_min': 1744,
 'sessions_max': 1780,
 'gates_passed': 7,
 'largest_effect_bps': 6.305,
 'pairs_effect_below_cost': 6}

## Results

### 13.1 The seven verdicts

One frozen protocol, seven pairs, four pre-registrations written before the corresponding results existed
(D-010 + amendments A1/A2, D-012, D-014, D-016).

In [3]:
verdicts = table[["pair", "segment", "anchor", "bars", "sessions", "gate",
                  "s1_positive_cells", "best_effect_bps", "best_t",
                  "round_trip_cost_bps", "effect_over_cost", "verdict",
                  "decision", "experiment"]].copy()
verdicts["s1_positive_cells"] = verdicts["s1_positive_cells"].astype(str) + "/20"
verdicts["vs_cost"] = [f"{r:.1f}x above" if r >= 1 else f"{1 / r:.1f}x below"
                       for r in verdicts["effect_over_cost"]]
print(verdicts.drop(columns=["effect_over_cost"]).to_string(index=False))

   pair        segment    anchor   bars  sessions gate s1_positive_cells  best_effect_bps  best_t  round_trip_cost_bps                    verdict decision experiment     vs_cost
MES_MYM     S&P-vs-Dow      unit 676560      1744 PASS              0/20            1.410    1.93                2.000            A-006 FALSIFIED    D-011    EXP-008  1.4x below
MES_MNQ  S&P-vs-Nasdaq      unit 684300      1780 PASS              0/20            1.781    2.84                2.000            A-006 FALSIFIED    D-013    EXP-009  1.1x below
MES_M2K S&P-vs-Russell      unit 684300      1780 PASS              8/20            6.305    5.31                2.000 AMBIGUOUS / MICROSTRUCTURE    D-015    EXP-010  3.2x above
  ZF_ZN          5s10s vol_ratio 675840      1746 PASS             20/20            0.286    6.59                3.103     AMBIGUOUS / IMMATERIAL    D-017    EXP-011 10.9x below
  ZT_ZF           2s5s vol_ratio 675840      1746 PASS             20/20            0.202    6.88             

`s1_positive_cells` is criterion (a): cells of the 4×5 entry/horizon grid that are positive with
session-clustered t ≥ 3 in the estimation-free specification. `cost_multiple` is how many times the round-trip
cost exceeds the largest effect either look-ahead-safe specification produced.

Read the two columns together. The index pairs fail criterion (a) outright and their significant cells point
at *continuation*. The Treasury pairs satisfy criterion (a) about as strongly as it can be satisfied — and are
7–11× too small to trade. MES–M2K is the only pair whose best cell clears its own cost, and it is the pair the
base-sampling check disqualified.

### 13.2 Why significance was never the binding constraint

In [4]:
econ = table[["pair", "best_effect_bps", "best_t", "round_trip_cost_bps", "cost_basis"]].copy()
econ["criterion_a"] = np.where(table["s1_positive_cells"] > 0, "satisfied", "FAILS")
print(econ.to_string(index=False))
print()
print(f"largest effect anywhere in the program : {totals['largest_effect_bps']:.3f} bps")
print(f"pairs whose effect clears its own cost : {totals['pairs_effect_clears_cost']} of {totals['pairs_tested']}")
print(f"pairs returning REVERSION PRESENT      : {totals['pairs_reversion_present']} of {totals['pairs_tested']}")

   pair  best_effect_bps  best_t  round_trip_cost_bps                                                      cost_basis criterion_a
MES_MYM            1.410    1.93                2.000 report 02 §5 band 2-3 bps incl. A-007 placeholder; low end used       FAILS
MES_MNQ            1.781    2.84                2.000 report 02 §5 band 2-3 bps incl. A-007 placeholder; low end used       FAILS
MES_M2K            6.305    5.31                2.000 report 02 §5 band 2-3 bps incl. A-007 placeholder; low end used   satisfied
  ZF_ZN            0.286    6.59                3.103             spec-derived (A-003 ticks, beta=0.583); spread only   satisfied
  ZT_ZF            0.202    6.88                1.415             spec-derived (A-003 ticks, beta=0.449); spread only   satisfied
  ZN_ZB            0.664    6.49                5.853             spec-derived (A-003 ticks, beta=0.554); spread only   satisfied
  ZT_ZN            0.166    4.07                1.504             spec-derived (A-003 tick

### 13.3 The pre-committed base-sampling check, all seven pairs

Criterion (b): a genuine reversion process does not care how the clock is sliced, so its variance ratio must
survive coarser base bars. Microstructure noise does not scale with the sampling interval, so it must not.
The check was declared decisive in D-014 and again in D-016 for all four Treasury pairs — before any result.

In [5]:
walk = table[["pair", "vr_res_1min", "vr_res_5min", "vr_res_15min", "vr_walk_x",
              "leg_a_vr_q2", "leg_b_vr_q2"]].copy()
print(walk.to_string(index=False))
print()
print("leg VR above 1 at q=2 = LAGGED price adjustment, not bounce:")
for pair in PAIRS:
    legs = leg_vr(read_variance_ratio(pair, MR), q=2)
    flagged = [f"{name} {v:.3f}" for name, v in legs.items() if v > 1.0]
    if flagged:
        a, b = pair.split("_")
        print(f"  {pair}: " + ", ".join(s.replace("LEGA", a).replace("LEGB", b)
                                        for s in flagged))

   pair  vr_res_1min  vr_res_5min  vr_res_15min  vr_walk_x  leg_a_vr_q2  leg_b_vr_q2
MES_MYM        0.756        0.852         0.950       1.26        0.989        0.990
MES_MNQ        0.928        0.941         0.983       1.06        0.988        1.017
MES_M2K        0.813        0.897         0.956       1.18        0.988        1.012
  ZF_ZN        0.134        0.467         0.827       6.17        0.889        0.818
  ZT_ZF        0.241        0.623         0.904       3.76        0.873        0.889
  ZN_ZB        0.208        0.594         0.893       4.28        0.818        0.868
  ZT_ZN        0.322        0.712         0.963       2.99        0.873        0.818

leg VR above 1 at q=2 = LAGGED price adjustment, not bounce:
  MES_MNQ: MNQ 1.017
  MES_M2K: M2K 1.012


### 13.4 The four near-miss fake edges

Each of these produced a publishable-looking result. Each was caught by a check that was in the battery
*before* the number existed. They are the program's most transferable output.

In [6]:
mym = read_variance_ratio("MES_MYM", MR)
res = mym[(mym.series == "RES1") & (mym.base_step_min == 1)].set_index("q")["vr"]
bounce = {"headline": f"MES-MYM residual VR(30) = {res.loc[30]:.3f}, bootstrap p < 0.001",
          "tell_shape": f"VR(120)/VR(30) = {res.loc[120] / res.loc[30]:.3f} (flat floor, not ~1/q decay)",
          "tell_base": f"at 5-min base bars VR walks to {matched_elapsed_vr(mym)[5]:.3f}"}

m2k_legs = leg_vr(read_variance_ratio("MES_M2K", MR), q=2)
m2k_best = largest_honest_effect(read_conditional("MES_M2K", MR))
leadlag = {"headline": f"MES-M2K {m2k_best['mean_session_bps']:+.2f} bps at t = {m2k_best['t_clustered']:.2f}, monotone in entry_z, clears cost",
           "tell_leg": f"M2K's own VR at q=2 = {m2k_legs['LEGB']:.3f} (ABOVE 1 = lagged adjustment)",
           "tell_base": f"residual VR {matched_elapsed_vr(read_variance_ratio('MES_M2K', MR))[1]:.3f} -> {matched_elapsed_vr(read_variance_ratio('MES_M2K', MR))[15]:.3f} as bars coarsen"}

sliding = {"headline": "MES-MYM S2 reported +2.14 bps at t = +8.91 before amendment A2",
           "tell": "differencing a trailing-mean residual credits the REFERENCE WINDOW sliding; on two "
                   "independent random walks the same code reports +5.4 bps at t = 20.3",
           "after_fix": "-0.32 bps at t = -0.67 (S1 and S3 reproduced exactly = the change was a correction)"}

zfzn = read_variance_ratio("ZF_ZN", MR)
quant = {"headline": f"ZF-ZN residual VR = {matched_elapsed_vr(zfzn)[1]:.3f} at 1-min sampling",
         "tell_base": f"walks to {matched_elapsed_vr(zfzn)[15]:.3f} at 15-min bars ({matched_elapsed_vr(zfzn)[15] / matched_elapsed_vr(zfzn)[1]:.1f}x)",
         "tell_cost": "the same coarse ticks that manufacture the VR are what make the cost 3.1 bps"}

for name, ev in [("1. bid-ask bounce", bounce), ("2. lead-lag from a thin leg", leadlag),
                 ("3. sliding reference window", sliding), ("4. tick quantisation", quant)]:
    print(name)
    for k, v in ev.items():
        print(f"    {k:12s} {v}")
    print()

1. bid-ask bounce
    headline     MES-MYM residual VR(30) = 0.756, bootstrap p < 0.001
    tell_shape   VR(120)/VR(30) = 0.991 (flat floor, not ~1/q decay)
    tell_base    at 5-min base bars VR walks to 0.852

2. lead-lag from a thin leg
    headline     MES-M2K +6.30 bps at t = 5.31, monotone in entry_z, clears cost
    tell_leg     M2K's own VR at q=2 = 1.012 (ABOVE 1 = lagged adjustment)
    tell_base    residual VR 0.813 -> 0.956 as bars coarsen

3. sliding reference window
    headline     MES-MYM S2 reported +2.14 bps at t = +8.91 before amendment A2
    tell         differencing a trailing-mean residual credits the REFERENCE WINDOW sliding; on two independent random walks the same code reports +5.4 bps at t = 20.3
    after_fix    -0.32 bps at t = -0.67 (S1 and S3 reproduced exactly = the change was a correction)

4. tick quantisation
    headline     ZF-ZN residual VR = 0.134 at 1-min sampling
    tell_base    walks to 0.827 at 15-min bars (6.2x)
    tell_cost    the same coa

### 13.5 Funnel and total-trials disclosure

In [7]:
funnel = pd.DataFrame([
    ("pairs in the locked universe (D-002)", 7),
    ("pairs with a passed data-acceptance gate", int(totals["gates_passed"])),
    ("pairs satisfying criterion (a) in a look-ahead-safe spec", int((table["s1_positive_cells"] > 0).sum())),
    ("... and also surviving the base-sampling check (b)", 0),
    ("... and also clearing round-trip cost", 0),
    ("pairs advancing to notebooks 04-12", 0),
], columns=["gate", "pairs"])
print(funnel.to_string(index=False))
print()
print(f"cells examined under one protocol: {totals['cells_examined']} "
      f"(7 pairs x 3 specs x 4 entry thresholds x 5 horizons)")
print("verdict rule (c) — the effect must strengthen with entry threshold rather than live in one cell —")
print("is what guards this count, and it is not what decided any pair: cost and base sampling did.")

out = Path("../reports/machine_readable/nb13_program_summary.csv")
table.to_csv(out, index=False)
funnel.to_csv(Path("../reports/machine_readable/nb13_funnel.csv"), index=False)
print(f"\nwrote {out.name} and nb13_funnel.csv")

                                                    gate  pairs
                    pairs in the locked universe (D-002)      7
                pairs with a passed data-acceptance gate      7
pairs satisfying criterion (a) in a look-ahead-safe spec      5
      ... and also surviving the base-sampling check (b)      0
                   ... and also clearing round-trip cost      0
                      pairs advancing to notebooks 04-12      0

cells examined under one protocol: 420 (7 pairs x 3 specs x 4 entry thresholds x 5 horizons)
verdict rule (c) — the effect must strengthen with entry threshold rather than live in one cell —
is what guards this count, and it is not what decided any pair: cost and base sampling did.

wrote nb13_program_summary.csv and nb13_funnel.csv


### 13.6 What the program built and validated anyway

A negative verdict on the hypothesis is not a negative verdict on the machinery. Three things survive the
conclusion and are reusable by any successor program:

- **The D-009 own-splice constructor**, validated on all eight instruments and demonstrated deterministic
  twice (D-014 predicted M2K's flagged roll and its numbers in advance; ZT230831 flagged identically in both
  ZT runs). It replaces QuantConnect's adjustment, which is FALSIFIED — A-004, 40 of 220 bad factors.
- **The battery** (`intraday_reversion.py`, `pair_minute_report.py`), whose test suite encodes known answers:
  random walk → VR 1, planted AR(1) → closed-form VR and recovered half-life, planted bounce → must not read
  as an edge.
- **The three-part discipline that caught all four fake edges**: measure the position's P&L rather than the
  residual's change; compare against both legs' own variance ratios; confirm the result survives coarser base
  sampling.

## Limitations

- **A-007 / A-008 (costs) remain UNVERIFIED placeholders.** The Treasury conclusion is robust to a threefold
  cost error (the shortfall is 7–11×); the index conclusion does not rest on costs at all, since the sign is
  wrong before cost. A real broker schedule is still owed.
- **A-012 (CTD / delivery-cycle contamination in ZB/ZN) is UNVERIFIED.** ZN–ZB was disqualified on cost and
  microstructure before A-012 could bind.
- **A-013 (RTH-only captures the signal) is untested**, and is a live limitation for Treasuries specifically,
  whose liquid session starts around 08:20 ET.
- **L-013 (the 390-bar z-window spans the overnight break)** puts 22.7–24.7% of index signals in the first
  30 minutes. It is absent in Treasuries (12.7–13.6%), so it is an equity-session-open artifact of the signal
  configuration. It was deliberately left unfixed so that all seven pairs stayed comparable; notebook 06 owns
  the fix, and it changes the SIGNAL definition, not the hypothesis.
- **L-012 (MYM 2019-12-12 boundary print)** is a watch item, not a resolved question.
- **One window, one resolution, one venue.** Every result here is minute resolution, trade bars, equity RTH,
  2019-06 → 2026-04 with a free-tier end clip. The window is SPENT for this hypothesis: re-running the same
  grid on the same data after seeing these results would be a multiple-testing violation.

## Decision

**No-go for Version 1 as an intraday relative-value reversion program (D-019).**

All seven pairs in the locked universe were tested at minute resolution under one frozen, pre-registered
protocol. Not one produced a tradable intraday reversion result: two index pairs falsified with significant
*continuation*, one index pair unresolved with a lead-lag signature, and four Treasury pairs statistically
overwhelming but 7–11× below the cost of harvesting them. Under CLAUDE.md hard gate 4 this is a legitimate
research outcome and is recorded as the finding.

**What is NOT concluded:** that these markets contain no structure. They contain a real, consistent,
correctly-signed reversion effect. It is simply smaller than the tick, and most of what looked large was
microstructure.

Three open threads remain, each a NEW program requiring fresh pre-registration, none of which may reuse this
window's results as evidence: the L-013 session-anchored z-score (notebook 06), the MES–M2K delayed-entry test
(D-015), and a different resolution or venue (quote data, a Treasury-native 08:20 ET session, second/tick bars).

## What this means for the algorithm

Nothing proceeds to `lean/algorithm/` without the authorization token — regardless of how good the research
looks, and there is nothing here that looks good. Notebooks 04–12 as originally scoped are moot for Version 1
in their current form: there is no candidate to select a hedge for, size, cost-model or walk-forward.

The honest recommendation is to fund one of the three open threads, or to stop.